# 🚀 SVAMITVA Unified DGX Pipeline (V3)
### Professional Performance Optimization for DGX systems (8x A100/H100)

This notebook provides a production-grade entry point for training the full SVAMITVA feature extraction suite on high-performance compute.

### 🎯 Performance Highlights:
- **Dynamic GPU Selection**: Auto-detects and uses the best available GPU(s).
- **AMP Enabled**: Mixed-precision training for 2-3x speedup.
- **High-Throughput IO**: Optimized workers (32) and batch sizes (128).

**Target Accuracy:** ≥95% IoU.

In [ ]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7" # Uncomment to force specific GPUs
import sys
import torch
import logging
from pathlib import Path
import importlib
import shutil

# ── Environment Setup ──
MANUAL_REPO_ROOT = None

def find_repo_root(start_path, markers=["train.py", "models"], max_depth=5):
    curr = Path(start_path).resolve()
    for _ in range(max_depth):
        for marker in markers:
            if (curr / marker).exists():
                return curr
        curr = curr.parent
    return Path(start_path).resolve()

if MANUAL_REPO_ROOT:
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    try:    REPO_ROOT = find_repo_root(os.getcwd())
    except: REPO_ROOT = Path(".").resolve()

os.chdir(str(REPO_ROOT))

repo_path_str = str(REPO_ROOT)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)
importlib.invalidate_caches()

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
logging.getLogger("train_engine").setLevel(logging.INFO)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"Total GPUs Detected: {torch.cuda.device_count()}")

In [ ]:
import urllib.request
from pathlib import Path

# ── Dependency Check & Model Download ──
needed_pkgs = ["tensordict", "hydra-core>=1.3.2", "iopath>=0.1.10", "ultralytics"]
try:
    import tensordict, ultralytics
    print("✅ Core packages present.")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "--user"] + needed_pkgs, check=True)

sam2_ckpt = REPO_ROOT / "check" / "sam2.1_hiera_tiny.pt"
if not sam2_ckpt.exists():
    sam2_ckpt.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading SAM2 weights to {sam2_ckpt}...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt", 
        sam2_ckpt
    )
else:
    print(f"✅ SAM2 weights found at {sam2_ckpt}")

In [ ]:
# ── DGX Training Configuration ──
from train_engine.config import TrainingConfig

# Optimizing for High-Performance Availability
config = TrainingConfig(
    train_dirs=[Path("data/MAP1")], 
    batch_size=128,              
    num_epochs=150,
    learning_rate=5e-4,          
    num_workers=32,              
    mixed_precision=True,        
    freeze_backbone_epochs=5,
    checkpoint_dir=Path("check"),  # <-- User requested 'check' folder
    sam2_checkpoint=REPO_ROOT / "check" / "sam2.1_hiera_tiny.pt"
)

print(f"Training Dirs: {[str(p) for p in config.train_dirs]}")
print(f"Global Batch Size: {config.batch_size}")
print(f"Checkpoints will save to: {config.checkpoint_dir.absolute()}")

In [ ]:
# ── Data Loaders & Multi-Task Setup ──
from data.dataset import create_dataloaders
from models.model import EnsembleSvamitvaModel
from models.losses import MultiTaskLoss
from train_engine.trainer import Trainer

train_loader, val_loader = create_dataloaders(
    train_dirs=config.train_dirs,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    val_split=0.15,
    image_size=config.tile_size
)

model = EnsembleSvamitvaModel(
    checkpoint_path=str(config.sam2_checkpoint),
    num_roof_classes=config.num_roof_classes,
    pretrained=True
)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=MultiTaskLoss(),
    config=config
)

print("✅ Pipeline Initialized. Multi-GPU will activate automatically if available.")

In [ ]:
# 🚀 Start Training
try:
    # Note: Trainer automatically selects the best available GPU(s)
    trainer.fit()
except KeyboardInterrupt:
    print("\n⏸️ Training paused by user.")
except Exception as e:
    print(f"\n❌ Training error: {e}")
    import traceback
    traceback.print_exc()

---
## 📊 Higher Efficiency Options (Terminal-Only)

To fully leverage NvLink/DDP for maximum speed on the DGX at 8x scale:

```bash
torchrun --nproc_per_node=8 train.py \
    --train_dirs ./data/MAP1 \
    --batch_size 128 --cache_features --checkpoint_dir check
```